# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [202]:
%pip install beautifulsoup4
%pip install faiss-cpu


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [203]:
from pathlib import Path
from bs4 import BeautifulSoup

documents = []

for path in Path("documents").glob("*.html"):
    html = path.read_text(encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    # Remove things we don't want embedded
    for tag in soup(["script", "style", "nav"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    documents.append({
        "title": path.stem,
        "text": text
    })

print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(doc["title"], len(doc["text"]))

Loaded 9 documents
Spring-Authorization 29689
Spring-Security 32542
Spring-Data-JPA 5514
Spring-Security-OAuth2 22775
Spring-Batch 11438
Spring-Data 8778
Spring-Dependencies 24266
Bean-Annotation 17495
Spring-JPA-Repository 8778


In [204]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

DOC = "\n\n".join(
    f"## {doc['title']}\n{doc['text']}"
    for doc in documents
)

print('corpus chars:', len(DOC))

corpus chars: 161477


In [205]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

configs = {'small (120/20)': chunk_fixed(DOC,120,20), 'large (320/40)': chunk_fixed(DOC,320,40), 'by-paragraph': chunk_paragraph(DOC)}
for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

small (120/20)   -> 1601 chunks
large (320/40)   -> 572 chunks
by-paragraph     -> 19 chunks


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [206]:
qa = [
    (
        "What does the @Bean annotation do?",
        "register a bean definition"
    ),
    (
        "What do GrantedAuthority objects represent in Spring Security?",
        "authorities that have been granted to the principal"
    ),
    (
        "Is Spring Batch a scheduling framework?",
        "Spring Batch is not a scheduling framework"
    ),
    (
        "What does Spring Data JPA provide?",
        "repository support for the Jakarta Persistence API"
    ),
    (
        "What is the central interface in the Spring Data repository abstraction?",
        "Repository"
    ),
    (
        "What are the two main types of dependency injection in Spring?",
        "constructor-based and setter-based dependency injection"
    ),
    (
        "What functionality do CrudRepository and ListCrudRepository provide?",
        "CRUD functionality for the entity class being managed"
    ),
    (
        "What is the base property prefix for OAuth Client registration properties?",
        "spring.security.oauth2.client.registration"
    ),
    (
        "What does SecurityFilterChain determine in Spring Security?",
        "which Spring Security Filter instances should be invoked for the current request"
    ),
    (
    "What is the purpose of DelegatingFilterProxy in Spring Security?",
    "delegate to a Filter implementation that is defined as a Spring Bean"
    )
]

indices = {n: build(c) for n, c in configs.items()}
# This example checks whether retrieved context contains the answer.
# It does not calculate the chunk-level precision and recall required below.
for n, c in configs.items():
    hits = sum(1 for q, fact in qa if fact.lower() in ' '.join(retrieve(indices[n], c, q)).lower())
    print(f'{n:16s} answer-recall@3: {hits}/{len(qa)} = {hits/len(qa):.2f}')


small (120/20)   answer-recall@3: 3/10 = 0.30
large (320/40)   answer-recall@3: 5/10 = 0.50
by-paragraph     answer-recall@3: 7/10 = 0.70


In [207]:
# Display the top retrieved chunks for each query and chunking configuration
for q, fact in qa:
    print("\n" + "=" * 80)
    print("QUESTION:", q)
    print("EXPECTED FACT:", fact)

    for name, chunks in configs.items():
        print(f"\n{name}")

        # Embed the question
        qe = embedder.encode(
            [q],
            normalize_embeddings=True
        ).astype("float32")

        # Get the top 10 chunk indexes for relevance judging
        _, chunk_ids = indices[name].search(qe, 10)

        for rank, chunk_id in enumerate(chunk_ids[0], 1):
            if chunk_id >= 0:
                print(f"\nRank {rank} - Chunk {chunk_id}")
                print(chunks[chunk_id])


QUESTION: What does the @Bean annotation do?
EXPECTED FACT: register a bean definition

small (120/20)

Rank 1 - Chunk 1346
ean annotation in a @Configuration -annotated or in a @Component -annotated class. Declaring a Bean To declare a bean, y

Rank 2 - Chunk 1449
ultiple names, otherwise known as bean aliasing. The name attribute of the @Bean annotation accepts a String array for t

Rank 3 - Chunk 1418
fying Bean Scope Spring includes the @Scope annotation so that you can specify the scope of a bean. Using the @Scope Ann

Rank 4 - Chunk 1457
erhaps through JMX) for monitoring purposes. To add a description to a @Bean , you can use the @Description annotation, 

Rank 5 - Chunk 1411
use for cross-reference calls in other @Bean methods that intend to refer to the provided resource here). In the case of

Rank 6 - Chunk 1182
asses annotated with @Component , @Controller , and so forth), or @Bean methods in Java-based @Configuration classes. Th

Rank 7 - Chunk 1383
ycle Callbacks Any clas

In [208]:
# Manual relevance labels for top-10 retrieved chunks
#
# 1 = chunk meaningfully helps answer the expected fact
# 0 = chunk does not meaningfully help answer the expected fact
#
# Each list corresponds to ranks 1 through 10.

relevance_labels = {

    "What does the @Bean annotation do?": {
        "small (120/20)": [
            1, 0, 0, 0, 0, 0, 0, 1, 1, 1
        ],
        "large (320/40)": [
            0, 0, 1, 1, 0, 0, 1, 0, 0, 0
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 0, 0, 0, 0
        ]
    },

    "What do GrantedAuthority objects represent in Spring Security?": {
        "small (120/20)": [
            0, 0, 0, 1, 0, 0, 0, 0, 0, 1
        ],
        "large (320/40)": [
            0, 0, 1, 0, 0, 0, 1, 0, 0, 1
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 0, 0, 0, 0
        ]
    },

    "Is Spring Batch a scheduling framework?": {
        "small (120/20)": [
            1, 1, 0, 1, 1, 0, 0, 0, 1, 1
        ],
        "large (320/40)": [
            1, 1, 1, 1, 0, 1, 0, 0, 0, 1
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 0, 0, 0, 0
        ]
    },

    "What does Spring Data JPA provide?": {
        "small (120/20)": [
            0, 0, 0, 0, 1, 0, 0, 0, 0, 0
        ],
        "large (320/40)": [
            1, 0, 0, 0, 0, 0, 0, 1, 0, 0
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 1, 1, 0, 0
        ]
    },

    "What is the central interface in the Spring Data repository abstraction?": {
        "small (120/20)": [
            1, 1, 1, 0, 0, 0, 1, 1, 0, 0
        ],
        "large (320/40)": [
            1, 1, 1, 0, 0, 0, 1, 0, 0, 0
        ],
        "by-paragraph": [
            1, 1, 0, 0, 0, 0, 1, 1, 0, 0
        ]
    },

    "What are the two main types of dependency injection in Spring?": {
        "small (120/20)": [
            1, 1, 0, 1, 0, 1, 0, 1, 0, 0
        ],
        "large (320/40)": [
            1, 1, 1, 1, 0, 0, 1, 0, 0, 1
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 1, 0, 0, 0, 0
        ]
    },

    "What functionality do CrudRepository and ListCrudRepository provide?": {
        "small (120/20)": [
            1, 1, 1, 1, 1, 1, 1, 0, 0, 1
        ],
        "large (320/40)": [
            1, 1, 0, 1, 1, 1, 1, 0, 0, 1
        ],
        "by-paragraph": [
            1, 1, 1, 1, 0, 0, 0, 0, 0, 0
        ]
    },

    "What is the base property prefix for OAuth Client registration properties?": {
        "small (120/20)": [
            1, 1, 0, 1, 0, 1, 0, 1, 1, 1
        ],
        "large (320/40)": [
            1, 1, 0, 1, 1, 1, 0, 0, 1, 0
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 0, 0, 0, 0
        ]
    },

    "What does SecurityFilterChain determine in Spring Security?": {
        "small (120/20)": [
            1, 1, 0, 0, 0, 0, 1, 0, 0, 0
        ],
        "large (320/40)": [
            1, 0, 1, 0, 0, 0, 0, 1, 1, 1
        ],
        "by-paragraph": [
            1, 0, 1, 0, 0, 0, 1, 0, 0, 0
        ]
    },

    "What is the purpose of DelegatingFilterProxy in Spring Security?": {
        "small (120/20)": [
            1, 1, 0, 0, 1, 1, 0, 1, 0, 0
        ],
        "large (320/40)": [
            0, 1, 1, 1, 1, 0, 1, 0, 1, 1
        ],
        "by-paragraph": [
            1, 0, 0, 0, 0, 0, 0, 0, 0, 0
        ]
    }
}

In [209]:
precision_results = {}

for question, configs_for_question in relevance_labels.items():
    precision_results[question] = {}

    for config_name, labels in configs_for_question.items():

        # Only evaluate the top 3 retrieved chunks
        top_3_labels = labels[:3]

        precision = sum(top_3_labels) / 3

        precision_results[question][config_name] = precision


for question, results in precision_results.items():
    print(f"\n{question}")

    for config_name, precision in results.items():
        print(f"  {config_name}: Precision@3 = {precision:.2f}")


What does the @Bean annotation do?
  small (120/20): Precision@3 = 0.33
  large (320/40): Precision@3 = 0.33
  by-paragraph: Precision@3 = 0.33

What do GrantedAuthority objects represent in Spring Security?
  small (120/20): Precision@3 = 0.00
  large (320/40): Precision@3 = 0.33
  by-paragraph: Precision@3 = 0.33

Is Spring Batch a scheduling framework?
  small (120/20): Precision@3 = 0.67
  large (320/40): Precision@3 = 1.00
  by-paragraph: Precision@3 = 0.33

What does Spring Data JPA provide?
  small (120/20): Precision@3 = 0.00
  large (320/40): Precision@3 = 0.33
  by-paragraph: Precision@3 = 0.33

What is the central interface in the Spring Data repository abstraction?
  small (120/20): Precision@3 = 1.00
  large (320/40): Precision@3 = 1.00
  by-paragraph: Precision@3 = 0.67

What are the two main types of dependency injection in Spring?
  small (120/20): Precision@3 = 0.67
  large (320/40): Precision@3 = 1.00
  by-paragraph: Precision@3 = 0.33

What functionality do CrudRepo

In [210]:
config_names = [
    "small (120/20)",
    "large (320/40)",
    "by-paragraph"
]

for config_name in config_names:
    scores = [
        precision_results[question][config_name]
        for question in precision_results
    ]

    average_precision = sum(scores) / len(scores)

    print(f"{config_name}: Average Precision@3 = {average_precision:.2f}")

small (120/20): Average Precision@3 = 0.57
large (320/40): Average Precision@3 = 0.67
by-paragraph: Average Precision@3 = 0.47


In [211]:
recall_results = {}

for question, configs_for_question in relevance_labels.items():
    recall_results[question] = {}

    for config_name, labels in configs_for_question.items():

        # Relevant chunks retrieved in the top 3
        relevant_in_top_3 = sum(labels[:3])

        # Total relevant chunks in our top-10 judgment pool
        total_relevant = sum(labels)

        if total_relevant == 0:
            recall = 0.0
        else:
            recall = relevant_in_top_3 / total_relevant

        recall_results[question][config_name] = recall


for question, results in recall_results.items():
    print(f"\n{question}")

    for config_name, recall in results.items():
        print(f"  {config_name}: Recall@3 = {recall:.2f}")


What does the @Bean annotation do?
  small (120/20): Recall@3 = 0.25
  large (320/40): Recall@3 = 0.33
  by-paragraph: Recall@3 = 1.00

What do GrantedAuthority objects represent in Spring Security?
  small (120/20): Recall@3 = 0.00
  large (320/40): Recall@3 = 0.33
  by-paragraph: Recall@3 = 1.00

Is Spring Batch a scheduling framework?
  small (120/20): Recall@3 = 0.33
  large (320/40): Recall@3 = 0.50
  by-paragraph: Recall@3 = 1.00

What does Spring Data JPA provide?
  small (120/20): Recall@3 = 0.00
  large (320/40): Recall@3 = 0.50
  by-paragraph: Recall@3 = 0.33

What is the central interface in the Spring Data repository abstraction?
  small (120/20): Recall@3 = 0.60
  large (320/40): Recall@3 = 0.75
  by-paragraph: Recall@3 = 0.50

What are the two main types of dependency injection in Spring?
  small (120/20): Recall@3 = 0.40
  large (320/40): Recall@3 = 0.50
  by-paragraph: Recall@3 = 0.50

What functionality do CrudRepository and ListCrudRepository provide?
  small (120/20

In [212]:
config_names = [
    "small (120/20)",
    "large (320/40)",
    "by-paragraph"
]

for config_name in config_names:
    scores = [
        recall_results[question][config_name]
        for question in recall_results
    ]

    average_recall = sum(scores) / len(scores)

    print(f"{config_name}: Average Recall@3 = {average_recall:.2f}")

small (120/20): Average Recall@3 = 0.33
large (320/40): Average Recall@3 = 0.42
by-paragraph: Average Recall@3 = 0.78


### Chunking Strategy Comparison

Small chunks had an average Precision@3 of 0.57 and an average Recall@3 of 0.33. This indicates that small chunks had difficulty ranking the relevant chunks highly enough to appear in the top three. This could be due to the complexity of the material and the need for multiple sentences to develop enough context.

Large chunks performed better, with an average Precision@3 of 0.67 and an average Recall@3 of 0.42. This was the highest Precision@3 and middle Recall@3 of the three strategies. This makes sense because 320 words may provide enough surrounding context for the embedding to capture the concept without splitting important information across too many chunks.

Paragraph chunking was interesting, with an average Precision@3 of 0.47 and an average Recall@3 of 0.78. This indicates that paragraph chunking was very good at surfacing relevant material in the top results, but the top three also contained more irrelevant chunks. The natural paragraph boundaries may provide enough context to identify relevant information, while also including information that is not directly relevant to the question.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [213]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

def gemini_chat(messages, model="gemini-3.6-flash", **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    #prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    prompt = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so.\n\n"
        "Context:\n- "
        + "\n- ".join(ctx)
        + f"\n\nQuestion: {q}\nAnswer:"
    )
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer(
    "What are the two main types of dependency injection in Spring?",
    config="by-paragraph"
))

Based on the provided context, the two major variants (types) of dependency injection are:

1. **Constructor-based dependency injection**
2. **Setter-based dependency injection**


## Part 4 and 5: failure, submit

In [214]:
# Part 4: Inspect a retrieval failure

failure_query = "What is the difference between constructor injection and setter injection?"

qe = embedder.encode(
    [failure_query],
    normalize_embeddings=True
).astype("float32")

print("QUERY:", failure_query)

for name, chunks in configs.items():
    print(f"\n--- {name} ---")

    # Retrieve top 3 chunks
    _, chunk_ids = indices[name].search(qe, 3)

    for rank, chunk_id in enumerate(chunk_ids[0], 1):
        if chunk_id >= 0:
            print(f"\nRank {rank} - Chunk {chunk_id}")
            print(chunks[chunk_id])

QUERY: What is the difference between constructor injection and setter injection?

--- small (120/20) ---

Rank 1 - Chunk 1224
to be configured by setters rather than constructors. Alternatively, avoid constructor injection and use setter injectio

Rank 2 - Chunk 1196
ed everywhere the code uses the dependency. One benefit of setter injection is that setter methods make objects of that 

Rank 3 - Chunk 1194
ration of concerns. Setter injection should primarily only be used for optional dependencies that can be assigned reason

--- large (320/40) ---

Rank 1 - Chunk 428
e for setter injection. Use the DI style that makes the most sense for a particular class. Sometimes, when dealing with third-party classes for which you do not have the source, the choice is made for you. For example, if a third-party class does not expose any setter methods, then constructor injection may be the only

Rank 2 - Chunk 426
ell, implying that the class likely has too many responsibilities and should be ref

## Part 4: Retrieval Failure

**Query:** "What is the difference between constructor injection and setter injection?"

**Retrieved Answer:** "to be configured by setters rather than constructors. Alternatively, avoid constructor injection and use setter injection..."

I examined the small (120/20) configuration and evaluated Rank 1. The answer was essentially discussing why you might use setter injection instead of constructor injection, rather than defining the difference between setter and constructor injection.

This is a retrieval failure because we were explicitly asking for the difference between setter and constructor injection and instead received a discussion about which one to choose. Although Rank 1 contains the terms setter injection and constructor injection, it does not actually compare them. This makes the result semantically related to the query, but not sufficient to answer it.

The smaller chunk size may have contributed to this problem because it provides less surrounding context. With only 120 words, relevant information can be separated from the additional sentences needed to fully explain a concept. This can result in chunks that are semantically related to the query but do not contain enough context to answer it.

To resolve this, a reranking step could take a larger set of initially retrieved chunks and rank them again based on how well they actually answer the query.